# RecurSieve example

Simulate a two-group dataset (2000 cells, 2000 genes) and run RecurSieve with default parameters.

In [1]:
import numpy as np
import pandas as pd
import anndata as ad
from recursieve import recursieve

## Simulate data

Background counts are negative binomial with a per-cell library size. The first 10 genes are staged like the bimodal scenario in `manuscript/feature_comparision.py`: `control` is unimodal, `disease` is bimodal around the same center, so group means match and differential expression has little to find. In `disease`, cells switch between two states that raise one half of the staged genes and lower the other, so the genes move together. Each cell keeps its staged-block total, so library size is preserved.

In [2]:
rng = np.random.default_rng(0)
n_cells, n_genes, n_staged = 2000, 2000, 10
center, sd, sep = 8.0, 1.5, 4.0

groups = np.repeat(["control", "disease"], n_cells // 2)
disease = groups == "disease"

# background
gene_means = rng.gamma(shape=0.6, scale=1.0, size=n_genes)
gene_means[:n_staged] = 5.0
library = rng.lognormal(mean=0.0, sigma=0.3, size=n_cells)
mu = library[:, None] * gene_means[None, :]
X = rng.poisson(rng.gamma(shape=10.0, scale=mu / 10.0))

# staged pattern: unimodal in control, coordinated bimodal in disease
pattern = rng.normal(center, sd, size=(n_cells, n_staged))
state = rng.random(n_cells) < 0.5
half = np.arange(n_staged) < n_staged // 2
shift = np.where(state[:, None] == half[None, :], sep, -sep)
pattern[disease] += shift[disease]

# redistribute each cell's staged-block total by the pattern
pattern = np.maximum(pattern, 0.0)
prob = pattern / pattern.sum(axis=1, keepdims=True)
totals = X[:, :n_staged].sum(axis=1)
X[:, :n_staged] = [rng.multinomial(t, p) for t, p in zip(totals, prob)]

adata = ad.AnnData(
    X=X.astype(np.float32),
    obs=pd.DataFrame({"condition": groups}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=[f"gene_{i}" for i in range(n_genes)]),
)
staged = list(adata.var_names[:n_staged])
adata

AnnData object with n_obs × n_vars = 2000 × 2000
    obs: 'condition'

## Run RecurSieve

In [3]:
model = recursieve(
    adata=adata,
    group1="control",
    group2="disease",
    field_name="condition",
)

Iteration 1
Iteration 2
flip_rate=0.1485
Iteration 3
flip_rate=0.0975
Iteration 4
flip_rate=0.0725
Iteration 5
flip_rate=0.0605
Iteration 6
flip_rate=0.0675
Iteration 7
flip_rate=0.1115
Iteration 8
flip_rate=0.1010
Iteration 9
flip_rate=0.1075
Iteration 10
flip_rate=0.0770
Iteration 11
flip_rate=0.0820
Iteration 12
flip_rate=0.1060
Iteration 13
flip_rate=0.0640
Iteration 14
flip_rate=0.0685
Iteration 15
flip_rate=0.0510
Iteration 16
flip_rate=0.0600
Iteration 17
flip_rate=0.0540
Iteration 18
flip_rate=0.0330
Iteration 19
flip_rate=0.0390
Iteration 20
flip_rate=0.0375
Iteration 21
flip_rate=0.0390
Iteration 22
flip_rate=0.0435
Iteration 23
flip_rate=0.0465
Iteration 24
flip_rate=0.0510
Iteration 25
flip_rate=0.0375
Iteration 26
flip_rate=0.0505
Iteration 27
flip_rate=0.0360
Iteration 28
flip_rate=0.0375
stopping: no new flip-rate minimum in 10 iterations


## Results

`genes` is the full selected panel; `unique_gene_panel` is the subset not called by differential expression.

In [4]:
print("Panel:", model.genes)
print("Unique to RecurSieve:", model.unique_gene_panel)
print("Staged genes recovered:", sorted(set(model.unique_gene_panel) & set(staged)))

Panel: ['gene_4', 'gene_3', 'gene_0', 'gene_1', 'gene_2', 'gene_8', 'gene_5', 'gene_6', 'gene_7', 'gene_869', 'gene_56', 'gene_627', 'gene_1744', 'gene_1077', 'gene_317', 'gene_767', 'gene_1631', 'gene_914', 'gene_198', 'gene_1233', 'gene_1534', 'gene_553', 'gene_1779', 'gene_690', 'gene_933', 'gene_1105', 'gene_457', 'gene_578']
Unique to RecurSieve: ['gene_3', 'gene_1', 'gene_2', 'gene_869', 'gene_56', 'gene_627', 'gene_1744', 'gene_1077', 'gene_317', 'gene_767', 'gene_1631', 'gene_914', 'gene_198', 'gene_1233', 'gene_1534', 'gene_1779', 'gene_690', 'gene_933', 'gene_1105', 'gene_457', 'gene_578']
Staged genes recovered: ['gene_1', 'gene_2', 'gene_3']
